# 🐱 MushukLens — mushuk jinsini rasmdan taniydigan sun'iy intellekt

**Mavzu:** 12 xil mushuk jinsini (Abyssinian, Bengal, Birman, Bombay, British Shorthair, Egyptian Mau, Maine Coon, Persian, Ragdoll, Russian Blue, Siamese, Sphynx) rasmdan aniqlash.

**Nima uchun bu qiziq?** Mushuk va it ajratish oson. Lekin *Ragdoll* bilan *Birman*, *Bombay* bilan *Russian Blue* ni ajratish inson uchun ham qiyin. Ya'ni bu oddiy 2 klassli masala emas, **fine-grained (nozik farqlarga asoslangan) klassifikatsiya**.

**Reja (topshiriq talablari bilan):**

| # | Talab | Qaysi bo'limda |
|---|-------|----------------|
| 1 | Ma'lumot (kamida 2 klass) | 2–3 bo'lim: Oxford-IIIT Pet dataset, 12 ta mushuk jinsi |
| 2 | ImageNet'da o'qitilgan model + fine-tuning | 5–6 bo'lim: EfficientNet-B0 |
| 3 | Train / validation bo'lish | 4 bo'lim: stratified 80/20 |
| 4 | Accuracy, F1, confusion matrix | 8 bo'lim |
| 5 | Yangi rasmlarda sinash | 11–12 bo'lim |

**Qo'shimcha (kreativ) qismlar:** Grad-CAM (model rasmning qayeriga qarayapti?), eng ko'p adashgan juftliklar tahlili, xato qilingan rasmlar galereyasi, Wikipedia'dan avtomatik yangi rasm olish va o'z rasmingizni yuklab sinash.

> ⚠️ **Runtime → Change runtime type → GPU (T4)** ni yoqing. CPU'da ham ishlaydi, lekin ancha sekin.

**Dataset:** [Oxford-IIIT Pet Dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/) (Parkhi va boshq., 2012) — 37 ta zot (25 it, 12 mushuk), ~200 rasm/zot. Biz faqat 12 ta mushuk jinsini olamiz (~2400 rasm). U `torchvision` orqali avtomatik yuklanadi (taxminan 800 MB).

## 1. Kutubxonalar va sozlamalar

In [2]:
import os, random, time, io, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from torchvision.datasets import OxfordIIITPet, ImageFolder

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Qurilma:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU (sekin bo'ladi)")
print("torch:", torch.__version__)

Qurilma: cpu | CPU (sekin bo'ladi)
torch: 2.15.0+cpu


## 2. Ma'lumotni yuklash: faqat mushuk jinslari

Dataset'ning ichida it va mushuk zotlari aralash. Biz **12 ta mushuk jinsini** ajratib olib, `papka_nomi = klass_nomi` ko'rinishida saqlaymiz (`ImageFolder` formati). Rasmlar kattaligi 400 pikselgacha kichraytiriladi, shunda o'qish tez bo'ladi.

Dataset'ning rasmiy `trainval` va `test` bo'limlarini birlashtiramiz va train/validation'ga **o'zimiz** ajratamiz (4-bo'lim).

In [ ]:
ROOT = Path("/content/data") if Path("/content").exists() else Path("./data")
RAW_DIR = ROOT / "raw"
CAT_DIR = ROOT / "cats"

CAT_BREEDS = {"Abyssinian", "Bengal", "Birman", "Bombay", "British Shorthair",
              "Egyptian Mau", "Maine Coon", "Persian", "Ragdoll", "Russian Blue",
              "Siamese", "Sphynx"}

if not CAT_DIR.exists():
    print("Yuklanmoqda (~800 MB), 1-3 daqiqa...")
    parts = [("trainval", OxfordIIITPet(RAW_DIR, split="trainval", download=True)),
             ("test",     OxfordIIITPet(RAW_DIR, split="test",     download=True))]

    missing = CAT_BREEDS - set(parts[0][1].classes)
    assert not missing, f"Bu nomlar dataset'da topilmadi: {missing}. Mavjud nomlar: {parts[0][1].classes}"

    saved = 0
    for split_name, ds in parts:
        for i, lab in enumerate(ds._labels):
            breed = ds.classes[lab]
            if breed in CAT_BREEDS:
                img, _ = ds[i]
                img = img.convert("RGB")
                img.thumbnail((400, 400))
                out = CAT_DIR / breed.replace(" ", "_")
                out.mkdir(parents=True, exist_ok=True)
                img.save(out / f"{split_name}_{i}.jpg", quality=92)
                saved += 1
    print("Saqlangan rasmlar:", saved)
else:
    print("Papka allaqachon tayyor:", CAT_DIR)

full_raw = ImageFolder(CAT_DIR)
class_names = [c.replace("_", " ") for c in full_raw.classes]
num_classes = len(class_names)
print(f"\nKlasslar soni: {num_classes}\nJami rasmlar: {len(full_raw)}")
print(class_names)

Yuklanmoqda (~800 MB), 1-3 daqiqa...


## 3. Ma'lumotni ko'z bilan ko'ramiz (EDA)

In [ ]:
counts = np.bincount(full_raw.targets, minlength=num_classes)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=class_names, y=counts, ax=ax, palette="viridis")
ax.set_title("Har bir jins bo'yicha rasmlar soni")
ax.set_ylabel("Rasmlar")
plt.xticks(rotation=40, ha="right")
plt.tight_layout(); plt.show()
print(f"Eng kam: {counts.min()}, eng ko'p: {counts.max()} -> balans yaxshi, class-weight kerak emas.")

# Har jinsdan bittadan tasodifiy rasm
fig, axes = plt.subplots(3, 4, figsize=(13, 10))
rng = np.random.RandomState(0)
for c, ax in enumerate(axes.ravel()):
    idxs = [i for i, t in enumerate(full_raw.targets) if t == c]
    img, _ = full_raw[rng.choice(idxs)]
    ax.imshow(img); ax.set_title(class_names[c], fontsize=12); ax.axis("off")
plt.suptitle("12 ta mushuk jinsi", fontsize=16, y=1.0)
plt.tight_layout(); plt.show()

## 4. Train / Validation bo'lish

`train_test_split(..., stratify=...)` har bir jinsning **bir xil ulushi** (80% / 20%) ikkala qismga tushishini ta'minlaydi.

**Augmentatsiya** faqat train'ga qo'llanadi (tasodifiy kesish, aylantirish, rang o'zgartirish), validation'da faqat standart resize + center-crop bo'ladi. ImageNet mean/std bilan normalizatsiya majburiy, chunki model shunday ma'lumotda o'qitilgan.

In [ ]:
IMG_SIZE = 224
BATCH = 32
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ColorJitter(0.25, 0.25, 0.2, 0.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

targets = np.array(full_raw.targets)
train_idx, val_idx = train_test_split(
    np.arange(len(targets)), test_size=0.2, stratify=targets, random_state=SEED)

train_ds = Subset(ImageFolder(CAT_DIR, transform=train_tf), train_idx)
val_ds   = Subset(ImageFolder(CAT_DIR, transform=val_tf),   val_idx)

NW = 2
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=NW, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64,    shuffle=False, num_workers=NW, pin_memory=True)

print(f"Train: {len(train_ds)} rasm | Validation: {len(val_ds)} rasm")

## 5. Modelni tayyorlash: EfficientNet-B0 + yangi "bosh" (head)

- `weights=IMAGENET1K_V1` — model 1000 klassli ImageNet'da o'qitilgan og'irliklar bilan yuklanadi (mushuklar ham ImageNet'da bor, shuning uchun past darajadagi belgilar bizga juda mos).
- Oxirgi `Linear(1280 → 1000)` qatlamini `Linear(1280 → 12)` ga almashtiramiz.

**Fine-tuning strategiyasi (2 bosqich):**
1. **Bosqich A — faqat head:** backbone muzlatiladi, faqat yangi qatlam o'rganadi. Bu yangi qatlamning tasodifiy og'irliklari "yaxshi" backbone'ni buzib yubormasligi uchun kerak.
2. **Bosqich B — butun model:** hammasi ochiladi, lekin backbone uchun **kichik** learning rate (1e-4), head uchun kattaroq (1e-3), plus cosine scheduler.

In [ ]:
weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
model = models.efficientnet_b0(weights=weights)

in_feats = model.classifier[1].in_features          # 1280
model.classifier[1] = nn.Linear(in_feats, num_classes)
model = model.to(device)

total = sum(p.numel() for p in model.parameters())
print(f"Jami parametrlar: {total/1e6:.2f} M | Yangi head: {in_feats*num_classes + num_classes:,}")
print(model.classifier)

## 6. O'qitish

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def run_epoch(model, loader, optimizer=None, freeze_backbone_bn=False):
    train = optimizer is not None
    model.train(train)
    if train and freeze_backbone_bn:
        model.features.eval()      # muzlatilgan backbone'da BatchNorm statistikasi o'zgarmasin
    loss_sum, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * x.size(0)
            correct  += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return loss_sum / n, correct / n

history = {"tr_loss": [], "tr_acc": [], "va_loss": [], "va_acc": [], "phase": []}

def fit(epochs, optimizer, scheduler=None, phase="A", freeze_bn=False):
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = run_epoch(model, train_loader, optimizer, freeze_backbone_bn=freeze_bn)
        vl, va = run_epoch(model, val_loader)
        if scheduler: scheduler.step()
        for k, v in zip(history, (tl, ta, vl, va, phase)): history[k].append(v)
        print(f"[{phase}] ep {ep:02d}/{epochs} | train loss {tl:.3f} acc {ta:.3f} | "
              f"val loss {vl:.3f} acc {va:.3f} | {time.time()-t0:.0f}s")

# ---- Bosqich A: faqat head ----
for p in model.features.parameters():
    p.requires_grad = False
opt_a = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
fit(epochs=4, optimizer=opt_a, phase="A", freeze_bn=True)

In [ ]:
# ---- Bosqich B: butun modelni ochamiz ----
for p in model.parameters():
    p.requires_grad = True

EPOCHS_B = 8
opt_b = torch.optim.AdamW([
    {"params": model.features.parameters(),   "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 1e-3},
], weight_decay=1e-4)
sched_b = torch.optim.lr_scheduler.CosineAnnealingLR(opt_b, T_max=EPOCHS_B)
fit(epochs=EPOCHS_B, optimizer=opt_b, scheduler=sched_b, phase="B")

Path("/content/model").mkdir(exist_ok=True) if Path("/content").exists() else None
SAVE_PATH = "/content/model/mushuklens_effnet_b0.pt" if Path("/content").exists() else "mushuklens_effnet_b0.pt"
torch.save({"state_dict": model.state_dict(), "classes": class_names}, SAVE_PATH)
print("Model saqlandi:", SAVE_PATH)

## 7. O'qitish grafiklari

Vertikal chiziq — A bosqichdan B bosqichga o'tgan joy. Train va validation egri chiziqlari orasidagi katta farq overfitting belgisi bo'lardi.

In [ ]:
ep = range(1, len(history["tr_loss"]) + 1)
n_a = history["phase"].count("A")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(ep, history["tr_loss"], "o-", label="train"); ax[0].plot(ep, history["va_loss"], "o-", label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch")
ax[1].plot(ep, history["tr_acc"], "o-", label="train"); ax[1].plot(ep, history["va_acc"], "o-", label="val")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch")
for a in ax:
    a.axvline(n_a + 0.5, color="gray", ls="--"); a.legend()
ax[0].text(n_a + 0.6, ax[0].get_ylim()[1]*0.92, "butun model ochildi →", fontsize=9, color="gray")
plt.tight_layout(); plt.show()

## 8. Natija: validation accuracy, F1, confusion matrix

Eslatma: bu yerda **oxirgi epoxadagi** model baholanadi. Eng yaxshi epoxani validation bo'yicha tanlamaymiz, chunki bunday tanlov validation natijasini sun'iy ravishda yaxshilab ko'rsatadi.

In [ ]:
@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    ys, ps, probs = [], [], []
    for x, y in loader:
        out = model(x.to(device))
        pr = F.softmax(out, 1).cpu()
        ys.append(y); ps.append(pr.argmax(1)); probs.append(pr)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy(), torch.cat(probs).numpy()

y_true, y_pred, y_prob = predict_loader(model, val_loader)

acc      = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average="macro")
f1_wt    = f1_score(y_true, y_pred, average="weighted")

print("="*50)
print(f"Validation accuracy : {acc*100:.2f}%")
print(f"F1 (macro)          : {f1_macro:.4f}")
print(f"F1 (weighted)       : {f1_wt:.4f}")
print("="*50, "\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax[0], cbar=False)
ax[0].set_title("Confusion matrix (soni)")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Oranges", xticklabels=class_names, yticklabels=class_names, ax=ax[1], cbar=False)
ax[1].set_title("Confusion matrix (normallashgan: har qator = 1)")
for a in ax:
    a.set_xlabel("Model bashorati"); a.set_ylabel("Haqiqiy jins")
    plt.setp(a.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout(); plt.show()

# Eng ko'p adashilgan juftliklar
pairs = [(cm[i, j], class_names[i], class_names[j]) for i in range(num_classes) for j in range(num_classes) if i != j and cm[i, j] > 0]
pairs.sort(reverse=True)
print("🔎 Eng ko'p adashilgan juftliklar (haqiqiy → model aytgan):")
for c, a, b in pairs[:6]:
    print(f"  {a:18s} → {b:18s}: {c} marta")

## 9. Model qayerda xato qildi? (xatolar galereyasi)

In [ ]:
wrong = np.where(y_true != y_pred)[0]
print(f"Validation'da xatolar: {len(wrong)} / {len(y_true)}")

if len(wrong):
    # modelning eng ishonch bilan aytgan xatolari birinchi
    wrong = wrong[np.argsort(-y_prob[wrong].max(1))][:8]
    fig, axes = plt.subplots(2, 4, figsize=(15, 8))
    for ax, k in zip(axes.ravel(), wrong):
        img, _ = full_raw[val_idx[k]]
        ax.imshow(img); ax.axis("off")
        ax.set_title(f"Haqiqiy: {class_names[y_true[k]]}\nModel: {class_names[y_pred[k]]} ({y_prob[k].max()*100:.0f}%)",
                     fontsize=10, color="crimson")
    plt.suptitle("Modelning eng 'ishonchli' xatolari", fontsize=15)
    plt.tight_layout(); plt.show()

## 10. Grad-CAM: model rasmning qayeriga qarayapti?

Grad-CAM oxirgi konvolyutsion qatlamdagi aktivatsiyalarni, tanlangan klass uchun gradient bilan og'irlab, issiqlik xaritasi hosil qiladi. Agar model mushukning **yuzi va yungiga** qarasa, yaxshi. Agar **fon**ga qarasa, u "shortcut" o'rganib olgan bo'lishi mumkin.

In [ ]:
inv_norm = transforms.Normalize(mean=[-m/s for m, s in zip(MEAN, STD)], std=[1/s for s in STD])

def grad_cam(model, x, class_idx=None):
    """x: (3,H,W) normallashgan tensor. Qaytaradi: (cam[H,W] 0..1, probs, class_idx)"""
    model.eval()
    store = {}
    def fwd_hook(module, inp, out):
        store["a"] = out
        out.register_hook(lambda g: store.__setitem__("g", g))
    h = model.features[-1].register_forward_hook(fwd_hook)
    try:
        with torch.enable_grad():
            xb = x.unsqueeze(0).to(device).requires_grad_(True)
            logits = model(xb)
            probs = F.softmax(logits, 1)[0].detach().cpu().numpy()
            idx = int(logits.argmax(1)) if class_idx is None else class_idx
            model.zero_grad()
            logits[0, idx].backward()
    finally:
        h.remove()
    a, g = store["a"].detach(), store["g"].detach()
    w = g.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * a).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.cpu().numpy(), probs, idx

def tensor_to_img(x):
    return inv_norm(x).clamp(0, 1).permute(1, 2, 0).cpu().numpy()

# Validation'dan 6 ta tasodifiy rasm
rng = np.random.RandomState(1)
sel = rng.choice(len(val_ds), 6, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(18, 6.5))
for col, k in enumerate(sel):
    x, y = val_ds[k]
    cam, probs, pred = grad_cam(model, x)
    img = tensor_to_img(x)
    axes[0, col].imshow(img); axes[0, col].axis("off")
    axes[0, col].set_title(f"Haqiqiy: {class_names[y]}", fontsize=10)
    axes[1, col].imshow(img); axes[1, col].imshow(cam, cmap="jet", alpha=0.45); axes[1, col].axis("off")
    ok = pred == y
    axes[1, col].set_title(f"{class_names[pred]} ({probs[pred]*100:.0f}%)", fontsize=10, color="green" if ok else "crimson")
plt.suptitle("Grad-CAM: yuqori qator — rasm, pastki qator — model e'tibori", fontsize=14)
plt.tight_layout(); plt.show()

## 11. Yangi rasmlarda sinov (modelga hech qachon ko'rsatilmagan)

Rasmlarni **Wikipedia**'ning tegishli maqolalaridan (*Siamese cat*, *Maine Coon*, ...) REST API orqali avtomatik olamiz. Bu rasmlar Oxford dataset'ida yo'q.

⚠️ **Halol eslatma:** "haqiqiy jins" yorlig'i maqola nomidan olinadi. Maqoladagi bosh rasm ba'zan boshqacha bo'lishi mumkin, shuning uchun bu mukammal "ground truth" emas. Kichik, taxminiy sinov sifatida qaralsin.

In [ ]:
HEADERS = {"User-Agent": "MushukLens-edu/1.0 (student ML assignment)"}
WIKI_TESTS = [("Siamese_cat", "Siamese"), ("Maine_Coon", "Maine Coon"), ("Sphynx_cat", "Sphynx"),
              ("Russian_Blue", "Russian Blue"), ("Bengal_cat", "Bengal"), ("Persian_cat", "Persian")]

def fetch_wiki_image(title):
    r = requests.get(f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}", headers=HEADERS, timeout=20)
    r.raise_for_status()
    j = r.json()
    url = (j.get("thumbnail") or j.get("originalimage") or {}).get("source")
    if not url:
        raise ValueError("Maqolada rasm yo'q")
    data = requests.get(url, headers=HEADERS, timeout=30)
    data.raise_for_status()
    return Image.open(io.BytesIO(data.content)).convert("RGB")

new_images = []   # (PIL, haqiqiy_jins yoki None, manba)
for title, label in WIKI_TESTS:
    try:
        new_images.append((fetch_wiki_image(title), label, f"Wikipedia: {title}"))
    except Exception as e:
        print(f"  ! {title} olinmadi: {e}")
print(f"Yuklangan yangi rasmlar: {len(new_images)}")

In [ ]:
def predict_pil(img, topk=3):
    x = val_tf(img)
    cam, probs, pred = grad_cam(model, x)
    top = np.argsort(-probs)[:topk]
    return x, cam, probs, top

def show_predictions(items):
    n = len(items)
    fig, axes = plt.subplots(n, 3, figsize=(14, 4.2 * n), squeeze=False,
                             gridspec_kw={"width_ratios": [1, 1, 1.1]})
    correct = total = 0
    for r, (img, label, src) in enumerate(items):
        x, cam, probs, top = predict_pil(img)
        base = tensor_to_img(x)
        axes[r, 0].imshow(img); axes[r, 0].axis("off")
        axes[r, 0].set_title(src if label is None else f"{src}\n(kutilgan: {label})", fontsize=9)
        axes[r, 1].imshow(base); axes[r, 1].imshow(cam, cmap="jet", alpha=0.45); axes[r, 1].axis("off")
        axes[r, 1].set_title("Grad-CAM", fontsize=10)
        names = [class_names[i] for i in top][::-1]
        vals = [probs[i] * 100 for i in top][::-1]
        colors = ["#4c9f70" if class_names[i] == label else "#7a8fa6" for i in top][::-1]
        axes[r, 2].barh(names, vals, color=colors); axes[r, 2].set_xlim(0, 100)
        axes[r, 2].set_title("Top-3 ehtimollik (%)", fontsize=10)
        for yy, v in enumerate(vals): axes[r, 2].text(v + 1, yy, f"{v:.1f}", va="center", fontsize=9)
        if label is not None:
            total += 1; correct += int(class_names[top[0]] == label)
    plt.tight_layout(); plt.show()
    if total: print(f"Yangi rasmlarda top-1 to'g'ri: {correct}/{total}")

show_predictions(new_images)

## 12. O'z rasmingizni sinab ko'ring (ixtiyoriy)

Colab'da quyidagi katakni ishga tushiring va mushuk rasmini yuklang. Agar Colab'da emas, lokal ishlayotgan bo'lsangiz, `MY_PATHS` ro'yxatiga fayl yo'llarini yozing.

In [ ]:
MY_PATHS = []   # lokal ishlatsangiz: ["mening_mushugim.jpg"]

try:
    from google.colab import files
    print("Mushuk rasmini tanlang (bekor qilish uchun oynani yoping):")
    up = files.upload()
    for name, data in up.items():
        MY_PATHS.append(io.BytesIO(data))
except Exception:
    pass

mine = []
for p in MY_PATHS:
    try:
        mine.append((Image.open(p).convert("RGB"), None, "Mening rasmim"))
    except Exception as e:
        print("Rasm ochilmadi:", e)

if mine:
    show_predictions(mine)
else:
    print("Rasm yuklanmadi, bu qism o'tkazib yuborildi.")

## 13. Yakuniy xulosa

In [ ]:
best_pair = pairs[0] if pairs else None
worst = (best_pair[1] + " → " + best_pair[2] + " (" + str(best_pair[0]) + " marta)") if best_pair else "xato yo'q"
print(f"""
╔══════════════════════════════════════════════════════════╗
  MODEL         : EfficientNet-B0 (ImageNet) + fine-tuning
  MA'LUMOT      : Oxford-IIIT Pet, 12 ta mushuk jinsi, {len(full_raw)} rasm
  BO'LISH       : train {len(train_ds)} / val {len(val_ds)} (stratified 80/20)
  VAL ACCURACY  : {acc*100:.2f}%
  F1 (macro)    : {f1_macro:.4f}
  ENG KO'P XATO : {worst}
╚══════════════════════════════════════════════════════════╝
""")

### Nimalar o'rgandik

- **Transfer learning kuchli:** ~1900 ta rasm bilan, bir necha daqiqada, noldan o'qitish bilan erishib bo'lmaydigan natijaga erishdik. ImageNet'dagi tayyor belgilar (qirralar, teksturalar, yuz shakllari) qayta ishlatiladi.
- **Ikki bosqichli fine-tuning** (avval head, keyin butun model kichik LR bilan) odatda barqarorroq ishlaydi.
- **Grad-CAM** modelning "nima uchun" shunday javob berganini ko'rsatadi, va bu faqat raqamga qaraganda ancha ko'p ma'lumot beradi.

### Cheklovlar (halol baho)

- Validation to'plami kichik (~480 rasm), shuning uchun natijadagi noaniqlik bir necha foizga teng bo'lishi mumkin. Bitta o'qitish natijasi (bitta random seed) mutlaq haqiqat emas.
- Dataset'dagi rasmlar nisbatan "toza" (yaxshi yoritilgan, mushuk markazda). Real hayotdagi xira, qisman yopilgan suratlarda aniqlik pastroq bo'ladi.
- Oxford Pets'dagi jinslar orasida ba'zan **chegara noaniq** (masalan, aralash zotlar), shuning uchun ba'zi "xatolar" aslida yorliq muammosi bo'lishi mumkin.
- Wikipedia'dan olingan yangi rasmlarda yorliq maqola nomiga tayanadi va bu qat'iy tekshirilgan emas.

### Keyingi qadamlar

1. Boshqa backbone bilan solishtirish: ResNet50, MobileNetV3, EfficientNet-B3.
2. K-fold cross-validation bilan ishonchli baho olish.
3. Test-time augmentation (TTA) va model ansambli.
4. Gradio yoki Streamlit orqali veb-ilova qilib chiqarish.